<a href="https://colab.research.google.com/github/arnav-subudhi/AI-Document-Processer/blob/main/RAG_Powered_Document_Analysis_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All required libraries and imports

In [ ]:
# open source model (Microsoft PHI2)
!pip install -q pypdf
!pip install -q python-dotenv
!pip install -q llama-index
!pip install -q llama-index-llms-huggingface
!pip install -q llama-index-embeddings-huggingface
!pip install -q gradio
!pip install einops
!pip install accelerate

!pip install -q -U huggingface-hub==0.34.0
from llama_index.core import VectorStoreIndex,SimpleDirectoryReader,ServiceContext
from llama_index.llms.huggingface import HuggingFaceLLM
import torch

!pip install -q PyMuPDF
import pymupdf as fitz # PyMuPDF
from llama_index.core import Document

from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.core import Settings

# metadata tagging
!pip install -q llama-index llama-index-readers-file llama-index-embeddings-huggingface transformers sentence-transformers
from llama_index.readers.file import PDFReader
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# doc classifier

!pip install -q PyPDF2
import pandas as pd
from PyPDF2 import PdfReader
import json

# route queries
!pip install -q llama-index-readers-file
!pip install anthropic

import time
import uuid
from anthropic import Anthropic

import json

from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

# rag pipeline
!pip install -q llama-index llama-index-llms-gemini pymupdf
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q llama-index llama-index-llms-anthropic pymupdf
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers
!pip install llama-index-llms-anthropic


from llama_index.llms.anthropic import Anthropic
import os
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nest_asyncio
from llama_index.core import Settings, VectorStoreIndex


Setting up LLM - Open Souce Microsoft PHI-2

In [ ]:
system_prompt = "You are a Q&A assistant. Your goal is to answer questions as accurately as possible based on the instructions and context provided."

# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = SimpleInputPrompt("<|USER|>{query_str}<|ASSISTANT|>")
llm = HuggingFaceLLM(
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={"temperature": 1.0, "do_sample": False},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="microsoft/phi-2",
    model_name="microsoft/phi-2",
    #device_map="cuda",
    # uncomment this if using CUDA to reduce memory usage
    model_kwargs={"dtype": torch.bfloat16}
)

PDF upload

In [ ]:
from google.colab import files
import os

def upload_pdf():
    """Upload a PDF file and return its path."""
    print("Please select a PDF file to upload:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.pdf'):
            # Save to the sample_docs directory
            pdf_path = os.path.join("sample_docs", filename)

            # Create directory if it doesn't exist
            os.makedirs("sample_docs", exist_ok=True)

            # Save the file
            with open(pdf_path, 'wb') as f:
                f.write(uploaded[filename])

            print(f"PDF saved to {pdf_path}")
            return pdf_path
        else:
            print(f"File {filename} is not a PDF. Please upload a PDF file.")

    return None

In [ ]:
pdf_path = upload_pdf()

Please select a PDF file to upload:


Saving pharma-blob-sample.pdf to pharma-blob-sample.pdf
PDF saved to sample_docs/pharma-blob-sample.pdf


In [ ]:
!pip install pypdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]
doc_pages

loader = PDFReader()
pages = loader.load_data(pdf_path)  # Returns one Document per page

# Print preview
print(f"Loaded {len(pages)} pages")
print(pages[0].text[:300])

Loaded 10 pages
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and


Determines if the previous document and current documents are the same. If so, they will be combined into a similar title, chunked, and then stored in vector database

In [ ]:
def is_same_document(prev_text, curr_text, doc_type=None):
    # Classify the current page
    curr_doc_type = classify_document_type(curr_text)

    # If the current page has the same document type/title as the
    # previous page, treat it as the same document
    if doc_type and curr_doc_type.strip().lower() == doc_type.strip().lower():
        return True

    prompt = f"""
You are determining whether two consecutive pages belong to the SAME
individual document in a pharmaceutical document package.

Rules:
- If the pages have the same document title or heading, they are the SAME document.
- If they have the same document number or reference number, they are the SAME document.
- If the second page continues the first page, they are the SAME document.
- "Page 2 of 2", "Page 3 of 4", "continued", etc. indicate the SAME document.
- Ignore OCR errors, formatting differences, capitalization, and whitespace.
- Only answer No when there is clear evidence that the second page starts
  a completely separate document.

Previous document type:
{doc_type or 'unknown'}

Previous page:
{prev_text}

Current page:
{curr_text}

Do these two pages belong to the SAME document?

Answer ONLY "Yes" or "No".
"""

    response = llm.complete(prompt).text.strip().lower()

    return response.startswith("yes")

Classifiying the document type

In [ ]:
def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
    {text}

    Respond with ONLY the document type name. No explanation.
    """
    response = llm.complete(prompt).text.strip().lower().replace(".", "")
    result = response.title()
    return result

In [ ]:
!pip install langchain-text-splitters


In [ ]:
import json

# ============================================================
# 1. Classify document type using Phi-2
# ============================================================

DOCUMENT_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other"
]


def classify_document_type(text):
    prompt = f"""
Classify this pharmaceutical document page into exactly ONE category.

Categories:
- Cover Letter
- Certificate of Quality
- Packaging Specification
- BSE/TSE Declaration
- Material Description
- Supplier Qualification
- Chain of Custody
- Other

Page Content:
{text}

Category:
"""

    response = llm.complete(prompt).text.strip()
    response_lower = response.lower()

    # Look for a valid document type in the model response
    for doc_type in DOCUMENT_TYPES:
        if doc_type.lower() in response_lower:
            return doc_type

    return "Other"


doc_type_array = [
    "Cover Letter",             # Page 1
    "Certificate of Quality",   # Page 2
    "Certificate of Quality",   # Page 3
    "Packaging Specification",  # Page 4
    "Packaging Specification",  # Page 5
    "BSE/TSE Declaration",      # Page 6
    "Material Description",     # Page 7
    "Supplier Qualification",   # Page 8
    "Supplier Qualification",   # Page 9
    "Chain of Custody"          # Page 10
]

# ============================================================
# 2. Determine document boundaries from document types
# ============================================================

results = []

page_in_doc = 0
previous_doc_type = None

for i, doc_type in enumerate(doc_type_array):

    print(f"Processing page {i + 1}/{len(doc_type_array)}...")

    # First page always starts a new document
    if i == 0:
        is_new_doc = "Yes"
        page_in_doc = 0

    # New document when the document type changes
    elif doc_type != previous_doc_type:
        is_new_doc = "Yes"
        page_in_doc = 0

    # Same document
    else:
        is_new_doc = "No"
        page_in_doc += 1

    results.append({
        "page": i,
        "is_new_doc": is_new_doc,
        "doc_type": doc_type,
        "page_in_doc": page_in_doc
    })

    previous_doc_type = doc_type


# ============================================================
# 3. Create LlamaIndex documents with metadata
# ============================================================

documents = []

for i, doc in enumerate(pages):

    doc.metadata = {
        "page_number": i + 1,
        "source_file": "pharma-blob-sample.pdf"
    }

    documents.append(doc)


# ============================================================
# 4. Manual document type array
# ============================================================

doc_type_array = [
    "Cover Letter",             # Page 1
    "Certificate of Quality",   # Page 2
    "Certificate of Quality",   # Page 3
    "Packaging Specification",  # Page 4
    "Packaging Specification",  # Page 5
    "BSE/TSE Declaration",      # Page 6
    "Material Description",     # Page 7
    "Supplier Qualification",   # Page 8
    "Supplier Qualification",   # Page 9
    "Chain of Custody"          # Page 10
]


# Add manual document type to metadata
for doc, doc_type in zip(documents, doc_type_array):
    doc.metadata["doc_type"] = doc_type




# ============================================================
# 6. Create embeddings and store in vector index
# ============================================================



# combines each of the similar pages into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)

all_documents = []

current_doc = []
current_doc_type = None
current_doc_id = None

for result, page in zip(results, doc_pages):

    if result["is_new_doc"] == "Yes":

        if current_doc:
            full_contract_text = "\n".join(current_doc)

            chunks = splitter.split_text(full_contract_text)

            for i, chunk in enumerate(chunks):
                all_documents.append(
                    Document(
                        text=chunk,
                        metadata={
                            "doc_type": current_doc_type,
                            "chunk_index": i,
                            "doc_id": current_doc_id,
                            "source_file": "pharma-blob-sample.pdf"
                        }
                    )
                )

        current_doc = []
        current_doc_type = result["doc_type"]
        current_doc_id = len(all_documents) + 1

    current_doc.append(page["text"])


# Process the last document
if current_doc:

    full_contract_text = "\n".join(current_doc)

    chunks = splitter.split_text(full_contract_text)

    for i, chunk in enumerate(chunks):
        all_documents.append(
            Document(
                text=chunk,
                metadata={
                    "doc_type": current_doc_type,
                    "chunk_index": i,
                    "doc_id": current_doc_id,
                    "source_file": "pharma-blob-sample.pdf"
                }
            )
        )



from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Settings

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(all_documents)




# ============================================================
# 7. Finished
# ============================================================

print("✅ Metadata added to pages")
print("✅ Embeddings created")
print("✅ Pages stored in vector index")


# ============================================================
# 8. Print results
# ============================================================

print("\n========== DOCUMENT RESULTS ==========\n")

for result in results:
    print(result)


Processing page 1/10...
Processing page 2/10...
Processing page 3/10...
Processing page 4/10...
Processing page 5/10...
Processing page 6/10...
Processing page 7/10...
Processing page 8/10...
Processing page 9/10...
Processing page 10/10...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Metadata added to pages
✅ Embeddings created
✅ Pages stored in vector index

========== DOCUMENT RESULTS ==========

{'page': 0, 'is_new_doc': 'Yes', 'doc_type': 'Cover Letter', 'page_in_doc': 0}
{'page': 1, 'is_new_doc': 'Yes', 'doc_type': 'Certificate of Quality', 'page_in_doc': 0}
{'page': 2, 'is_new_doc': 'No', 'doc_type': 'Certificate of Quality', 'page_in_doc': 1}
{'page': 3, 'is_new_doc': 'Yes', 'doc_type': 'Packaging Specification', 'page_in_doc': 0}
{'page': 4, 'is_new_doc': 'No', 'doc_type': 'Packaging Specification', 'page_in_doc': 1}
{'page': 5, 'is_new_doc': 'Yes', 'doc_type': 'BSE/TSE Declaration', 'page_in_doc': 0}
{'page': 6, 'is_new_doc': 'Yes', 'doc_type': 'Material Description', 'page_in_doc': 0}
{'page': 7, 'is_new_doc': 'Yes', 'doc_type': 'Supplier Qualification', 'page_in_doc': 0}
{'page': 8, 'is_new_doc': 'No', 'doc_type': 'Supplier Qualification', 'page_in_doc': 1}
{'page': 9, 'is_new_doc': 'Yes', 'doc_type': 'Chain of Custody', 'page_in_doc': 0}


Using an LLM to determine which document type is needed to answer the prompt


In [ ]:

VALID_DOC_TYPES = [
    "cover_letter", "certificate_of_quality", "packaging_specification",
    "bse_tse_declaration", "material_description", "supplier_qualification",
    "chain_of_custody", "unknown"
]

def clean_llm_label(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    # Check if the cleaned response contains a valid label
    for label in VALID_DOC_TYPES:
        if label in cleaned:
            return label
    # If no valid label is found within the response, return 'unknown'
    return "unknown"

def classify_query_llm(query, metadata_store):
    doc_list = "\n".join(
        [f"{i+1}. {doc.metadata['source_file']} - doc_type: {doc.metadata['doc_type']}" for i, doc in enumerate(metadata_store)]
    )

    prompt = f"""
  You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

  Available documents:
  {doc_list}

  User query: "{query}"

  Which document(s) are most likely to contain the answer?
  Respond with one of the following types:
  ["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
"material_description", "supplier_qualification", "chain_of_custody", "unknown"]


  Respond with ONLY the label. NO OTHER explanation.
  """

    response = llm.complete(prompt)
    return clean_llm_label(response.text)

In [ ]:
Settings.llm = llm

Expands the user's query to more querys

In [ ]:
def expand_query(query: str, num_expansions: int = 3) -> list:
    """Expand a query using Microsoft Phi-2."""

    prompt = f"""
You are helping search a pharmaceutical quality document.

Original query:
{query}

Generate {num_expansions} alternative search queries.

Requirements:
- Use different but related terminology.
- Include relevant pharmaceutical and quality terms.
- Keep each query focused on the same meaning as the original.
- Output ONLY the alternative queries.
- Put each query on a separate line.
"""

    response = llm.complete(prompt)

    expanded_queries = [
        line.strip()
        for line in response.text.split("\n")
        if line.strip()
    ]

    # Add original query
    if query not in expanded_queries:
        expanded_queries.insert(0, query)

    # Keep only the requested number of expansions + original
    return expanded_queries[:num_expansions + 1]

Combines the original query with the expanded queries to feed into the llm and pull more relevent chunks

In [ ]:
def combine_queries(original_query, expanded_queries):
    prompt = f"""
You are improving a search query for a pharmaceutical document retrieval system.

Original query:
{original_query}

Related query variations:
{chr(10).join(expanded_queries)}

Combine the useful information from these query variations into ONE
clear, focused search query.

Make sure that it includes key words from the original query.

Do not change the meaning of the original query.
Include important terminology that may help retrieve the relevant document.
Output ONLY the final search query. DO NOT EXPLAIN.
"""

    response = llm.complete(prompt)

    return response.text.strip()

If the doc type cannot be found with the llm, keyword search will brute force the output by using key words

In [ ]:

def keyword_search(query, documents):
    query = query.lower()
    keywords = ["part number", "article number", "item", "specification",
                "product", "certificate", "lot", "description",
                "sterilization", "BSE", "supplier"]

    for doc in documents:
        text = doc["text"].lower()
        if any(keyword.lower() in text for keyword in keywords):
            return doc
    return None


In [ ]:
import json

from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

Two cells below:

First one uses an LLM prompt to feed into the LLM, and then produce an answer

Second cell uses the built in LlamaIndex synthesizer, and then feeds it to the LLM to produce an answer

Only run either or cell -  expects to get top 4 most relevent chunks, and then uses that to feed into the open source LLM to get an answer

In [ ]:
# test query
start_query = "Were there any packaging configuration changes?"

# Expand query
expanded = expand_query(start_query)
query = combine_queries(start_query, expanded)

# Document type classification
DOC_TYPE_MAPPING = {
    "cover_letter": "Cover Letter",
    "certificate_of_quality": "Certificate of Quality",
    "packaging_specification": "Packaging Specification",
    "bse_tse_declaration": "BSE/TSE Declaration",
    "material_description": "Material Description",
    "supplier_qualification": "Supplier Qualification",
    "chain_of_custody": "Chain of Custody",
    "unknown": "Unknown"
}

predicted_doc_type = DOC_TYPE_MAPPING[
    classify_query_llm(query, all_documents)
]

if predicted_doc_type == "Unknown":
    predicted_doc_type = keyword_search(query, all_documents)

# Retrieve relevant chunks
retriever = index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="doc_type",
                value=predicted_doc_type,
                operator=FilterOperator.EQ
            )
        ]
    )
)

retrieved_nodes = retriever.retrieve(query)

# -----------------------------------
# Generate answer from retrieved chunks
# -----------------------------------

if retrieved_nodes:

    # Combine all retrieved chunks
    retrieved_text = "\n\n".join(
        node.node.get_content()
        for node in retrieved_nodes
    )

    prompt = f"""
Based only on the following pharmaceutical document excerpts, answer the user's question.

Document excerpts:
{retrieved_text}

User Question:
{start_query}

Provide a clear and specific answer based only on the information in the excerpts.
Include relevant dates, changes, or other specific details when available.
Do not discuss the retrieval process.
"""

    answer = llm.complete(prompt)

else:
    answer = "Could not find an answer because no relevant documents were retrieved."

# -----------------------------------
# Output
# -----------------------------------

output = {
    "query": start_query,
    "predicted_doc_type": predicted_doc_type,
    "matched_chunks": [
        {
            "text": node.node.get_content(),
            "metadata": node.node.metadata
        }
        for node in retrieved_nodes
    ],
    "answer": answer
}

print(json.dumps(output, indent=2))

Only run either top or bottom cell

In [ ]:
# test query and already given doc type

start_query = "Were there any packaging configuration changes?"
expanded = expand_query(start_query)
query = combine_queries(start_query, expanded)



# query = "Were there any packaging configuration changes?"
#predicted_doc_type = classify_query_llm(query, all_documents)


DOC_TYPE_MAPPING = {
    "cover_letter": "Cover Letter",
    "certificate_of_quality": "Certificate of Quality",
    "packaging_specification": "Packaging Specification",
    "bse_tse_declaration": "BSE/TSE Declaration",
    "material_description": "Material Description",
    "supplier_qualification": "Supplier Qualification",
    "chain_of_custody": "Chain of Custody",
    "unknown": "Unknown"
}

predicted_doc_type = DOC_TYPE_MAPPING[
    classify_query_llm(query, all_documents)
]

if predicted_doc_type == "Unknown":
  predicted_doc_type = keyword_search(query, all_documents)

# Retrieve relevant chunks
retriever = index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="doc_type",
                value=predicted_doc_type,
                operator=FilterOperator.EQ
            )
        ]
    )
)

retrieved_nodes = retriever.retrieve(query)


# Generate answer
synthesizer = get_response_synthesizer(
    response_mode=ResponseMode.COMPACT
)

response = synthesizer.synthesize(
    start_query,
    nodes=retrieved_nodes
)


# Create JSON output
output = {
    "query": start_query,
    "predicted_doc_type": predicted_doc_type,
    "matched_chunks": [
        {
            "text": node.node.get_content(),
            "metadata": node.node.metadata
        }
        for node in retrieved_nodes
    ],
    "answer": str(response.response)
}


# Print JSON
print(json.dumps(output, indent=2))